# Data Preprocessing

Let's load and pre-process some data with the [Pandas](https://github.com/pandas-dev/pandas) library. As always, we need to first install it with `pip`.

We assume you've already installed the packages in `requirements.txt` so we'll not repeat it here.

In [1]:
%pip install pandas==3.0.1


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2.2.1. Reading the Dataset

Let's write a sample CSV file with dummy data on house prices. It consists of the following columns.

1. `NumRooms`: number of rooms in the house
1. `RoofType`: the type of roof
1. `Price`: the price of the house in USD

In [2]:
import os

data_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(data_dir, exist_ok=True)
data_file = os.path.join(data_dir, 'house_tiny.csv')
data_csv = '''NumRooms,RoofType,Price
NA,NA,127500
2,NA,106000
4,Slate,178100
NA,NA,140000'''
with open(data_file, 'w') as f:
    f.write(data_csv)

Import the data with `pandas` and load the dataset with `read_csv`.

In [3]:
import pandas as pd

data = pd.read_csv(data_file)
print(data)

   NumRooms RoofType   Price
0       NaN      NaN  127500
1       2.0      NaN  106000
2       4.0    Slate  178100
3       NaN      NaN  140000


Let's split the input and target features with the `iloc` attribute.

1. The input features includes everything but the last column
1. The target features includes just the last column `Price` which is what we want to predict

We'll further use the `pd.get_dummies` method to convert the non-numerical categorical field `RoofType` to a collection of numerical fields. Each field in the collection represents a category, e,g, `RoofType_Slate`. If the field in the collection is `1` then the entry belongs to that category; otherwise it is `0`. This means for each entry, exactly 1 field in the collection is set to `1` with all others being `0`. This encoding is known as [one-hot encoding](https://en.wikipedia.org/wiki/One-hot).

Specify `dummy_na=True` in `pd.get_dummies` to generate an additional category `RoofType_nan` for entries with the roof type set to `NA`.

In [4]:
inputs, targets = data.iloc[:, :-1], data.iloc[:, -1]
inputs_onehot = pd.get_dummies(inputs, dummy_na=True)
inputs, inputs_onehot, targets

(   NumRooms RoofType
 0       NaN      NaN
 1       2.0      NaN
 2       4.0    Slate
 3       NaN      NaN,
    NumRooms  RoofType_Slate  RoofType_nan
 0       NaN           False          True
 1       2.0           False          True
 2       4.0            True         False
 3       NaN           False          True,
 0    127500
 1    106000
 2    178100
 3    140000
 Name: Price, dtype: int64)

2 items for improvement:

1. Notice the relationship between `RoofType_Slate` and `RoofType_nan`. When `RoofType_Slate` is `1`, `RoofType_nan` must be `0` and vice versa. In general, we can drop any 1 arbitrary column from the one-hot encoding representation without losing information. By convention, we drop the first column by specifying `drop_first=True`
1. `sklearn.OneHotEncoder` from [scikit-learn](https://scikit-learn.org/stable/) is preferable over `pd.get_dummies` as the former treats both training and test data consistently, while the latter may not correctly handle new categories appearing in test data only but not training data. This is not demonstrated below and left as an exercise to the reader

In [5]:
inputs_onehot_no_multicollinearity = pd.get_dummies(inputs, drop_first=True, dummy_na=True)
inputs_onehot_no_multicollinearity

,NumRooms,RoofType_nan
0,NaN,True
1,2.0,True
2,4.0,False
3,NaN,True


Let's treat the missing numerical values in `NumRooms` as well. An acceptable heuristic would be to replace the `NaN`s with the mean of the remaining entries in the same column. 

In [6]:
inputs_no_missing = inputs_onehot_no_multicollinearity.fillna(inputs_onehot_no_multicollinearity.mean())
inputs_no_missing

,NumRooms,RoofType_nan
0,3.0,True
1,2.0,True
2,4.0,False
3,3.0,True


## 2.2.3. Conversion to the Tensor Format

Now that all the features are purely numerical without `NaN` values, let's convert them to tensors. Pandas dataframes provide the `to_numpy` method and recall we can instantiate `mindspore.Tensor` with Numpy arrays directly.

In [7]:
import numpy as np

import mindspore
mindspore.set_device(device_target='Ascend', device_id=0)

X = mindspore.Tensor(inputs_no_missing.to_numpy(dtype=np.float32))
y = mindspore.Tensor(targets.to_numpy(dtype=np.float32))
X, y

/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/classifier/transdata/transdata_classifier.py:223: SyntaxWarning: invalid escape sequence '\B'
  Return BN\BH SCH Result
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:146: SyntaxWarning: invalid escape sequence '\c'
  2. In forward, tiling would not split c1 and c0, find c1\c0 based on t2.
/usr/local/Ascend/cann-8.5.0/python/site-packages/tbe/dsl/unify_schedule/vector/transdata/common/graph/transdata_graph_info.py:172: SyntaxWarning: invalid escape sequence '\c'
  1. Forward: tiling would not split c1\c0\h0, find c1\c0\h1\h0 based on t2
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangepiaipro-20t/lib/python3.12/site-packages/numpy/core/getlimits.py:549: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/home/HwHiAiUser/.pyenv/versions/3.12.13/envs/orangep

(Tensor(shape=[4, 2], dtype=Float32, value=
 [[ 3.00000000e+00,  1.00000000e+00],
  [ 2.00000000e+00,  1.00000000e+00],
  [ 4.00000000e+00,  0.00000000e+00],
  [ 3.00000000e+00,  1.00000000e+00]]),
 Tensor(shape=[4], dtype=Float32, value= [ 1.27500000e+05,  1.06000000e+05,  1.78100000e+05,  1.40000000e+05]))

## 2.2.4. Discussion

This chapter covered the basics of data loading and preprocessing but real-world data may require more complex data preprocessing pipelines. For example, the data may be spread across multiple CSV files, or stored in a relational database.

Handling outliers and faulty measurements is also a big topic. Ensure that the data is clean and properly handled before using it to train and evaluate your model. Your model can only be ever as good as the data it was provided in the first place.

## 2.2.5. Exercises

### 2.2.5.1. Iris dataset from UCI Machine Learning Repository

The [UCI Machine Learning Repository](https://archive.ics.uci.edu/) contains many sample datasets for exploring AI/ML algorithms and techniques. One such dataset is the [Iris](https://archive.ics.uci.edu/dataset/53/iris) dataset. It's a tiny dataset with just 150 instances and 4 input features.

A Python package `ucimlrepo` is available to install which contains the UCI Machine Learning Repository datasets. Alternatively, we can download and import the data directly from a zip file.

In [8]:
import urllib.request
import zipfile
from io import BytesIO
import os

iris_url = 'https://archive.ics.uci.edu/static/public/53/iris.zip'
iris_dir = os.path.join(os.getcwd(), 'iris')
with urllib.request.urlopen(iris_url) as response:
    with BytesIO(response.read()) as file:
        with zipfile.ZipFile(file, 'r') as contents:
            contents.extractall(path=iris_dir)

Let's load the CSV file `iris/iris.data` with Pandas. Since the file does not include headers by default, let's specify them explicitly in `pd.read_csv` based on the written information provided in `iris/iris.names`.

In [9]:
iris_csv = os.path.join(iris_dir, 'iris.data')
iris_headers =  ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']
df = pd.read_csv(iris_csv, header=None, names=iris_headers)
df

,sepal_length,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Iris-virginica
146,6.3,2.5,5.0,1.9,Iris-virginica
147,6.5,3.0,5.2,2.0,Iris-virginica
148,6.2,3.4,5.4,2.3,Iris-virginica


Let's apply some of the data preprocessing techniques we learned earlier.

1. Split input and target features by target name
1. Use one-hot encoding for the categories in the input features \(there are none\)
1. Use [label encoding](https://www.geeksforgeeks.org/machine-learning/ml-label-encoding-of-datasets-in-python/) for the target feature which is category-based

In [10]:
iris_inputs = df.drop(columns=['class'])
iris_target = df['class']
iris_inputs

,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [11]:
iris_target

0         Iris-setosa
1         Iris-setosa
2         Iris-setosa
3         Iris-setosa
4         Iris-setosa
            ...      
145    Iris-virginica
146    Iris-virginica
147    Iris-virginica
148    Iris-virginica
149    Iris-virginica
Name: class, Length: 150, dtype: str

In [12]:
iris_inputs_onehot = pd.get_dummies(iris_inputs, drop_first=True, dummy_na=True)
iris_inputs_onehot

,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [13]:
iris_target_label, iris_target_label_index = pd.factorize(iris_target)
iris_target_label

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [14]:
iris_target_label_index

Index(['Iris-setosa', 'Iris-versicolor', 'Iris-virginica'], dtype='str')

Let's check:

1. How many `NaN`s in the the input features
1. How many sentinel values `-1` in the target feature

In [15]:
iris_inputs_onehot_isna = iris_inputs_onehot.isna()
iris_inputs_onehot_isna

,sepal_length,sepal_width,petal_length,petal_width
0,False,False,False,False
1,False,False,False,False
2,False,False,False,False
3,False,False,False,False
4,False,False,False,False
...,...,...,...,...
145,False,False,False,False
146,False,False,False,False
147,False,False,False,False
148,False,False,False,False


In [16]:
iris_target_label_issentinel = iris_target_label == -1
iris_target_label_issentinel

array([False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False,
       False, False,

In [17]:
iris_num_instances = iris_inputs_onehot.shape[0]
sepal_length_nans = iris_inputs_onehot_isna['sepal_length'].sum()
sepal_width_nans = iris_inputs_onehot_isna['sepal_width'].sum()
petal_length_nans = iris_inputs_onehot_isna['petal_length'].sum()
petal_width_nans = iris_inputs_onehot_isna['petal_width'].sum()
iris_target_label_sentinels = iris_target_label_issentinel.sum()
print(f'Total number of instances: {iris_num_instances}')
print(f'Number of NaNs in column sepal_length: {sepal_length_nans}')
print(f'Number of NaNs in column sepal_width: {sepal_width_nans}')
print(f'Number of NaNs in column petal_length: {petal_length_nans}')
print(f'Number of NaNs in column petal_width: {petal_width_nans}')
print(f'Number of sentinels in target feature: {iris_target_label_sentinels}')

Total number of instances: 150
Number of NaNs in column sepal_length: 0
Number of NaNs in column sepal_width: 0
Number of NaNs in column petal_length: 0
Number of NaNs in column petal_width: 0
Number of sentinels in target feature: 0


Very clean data indeed! Since there is no missing data, we can convert them to tensors directly for our model training and evaluation.

In [18]:
X = mindspore.Tensor(iris_inputs_onehot.to_numpy())
y = mindspore.Tensor(iris_target_label)
X, y

(Tensor(shape=[150, 4], dtype=Float64, value=
 [[ 5.10000000e+00,  3.50000000e+00,  1.40000000e+00,  2.00000000e-01],
  [ 4.90000000e+00,  3.00000000e+00,  1.40000000e+00,  2.00000000e-01],
  [ 4.70000000e+00,  3.20000000e+00,  1.30000000e+00,  2.00000000e-01],
  ...
  [ 6.50000000e+00,  3.00000000e+00,  5.20000000e+00,  2.00000000e+00],
  [ 6.20000000e+00,  3.40000000e+00,  5.40000000e+00,  2.30000000e+00],
  [ 5.90000000e+00,  3.00000000e+00,  5.10000000e+00,  1.80000000e+00]]),
 Tensor(shape=[150], dtype=Int64, value= [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 
  0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 
  1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 
  1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 
  2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 
  2, 2, 2, 2, 2, 2]))

### 2.2.5.2. Handling large datasets

Large datasets that do not fit directly in memory should be loaded in _batches_, otherwise known as _chunks_. The `pd.read_csv` method has an optional `chunksize` parameter specifying the number of samples in each chunk.

We can then use a `for` loop to iterate through each chunk and perform mini-batch gradient descent, commonly known as mini-batch training.